## SVM Classifier
RBF-SVM with GridSearchCV and permutation feature importance.

In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (classification_report, confusion_matrix,
    roc_auc_score, roc_curve, accuracy_score,
    precision_score, recall_score, f1_score)
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

TRAIN_FEAT = "galaxy_features_train.csv"
TRAIN_LBLS = "trainlabel.csv"
TEST_FEAT  = "galaxy_features_test.csv"
TEST_LBLS  = "testlabel.csv"
CV_FOLDS   = 5
SEED       = 42


In [ ]:
def load_data(feat_csv, lbl_csv, name="Dataset"):
    feats = pd.read_csv(feat_csv)
    lbls  = pd.read_csv(lbl_csv).dropna(subset=['Label'])
    lbls['Label'] = lbls['Label'].astype(int)

    def name_from_file(fname):
        return fname.replace('norm_resized_','').replace('.fits','').replace('_',' ')
    feats['Target Name'] = feats['filename'].apply(name_from_file)

    merged = feats.merge(lbls, on='Target Name', how='inner')
    if len(merged) == 0:
        raise ValueError(f"No matches in {name}")
    print(f"{name}: {len(merged)} galaxies")
    feat_cols = [c for c in feats.columns if c not in ('filename','Target Name')]
    X = merged[feat_cols].values.astype(float)
    y = merged['Label'].values
    names = merged['Target Name'].values
    if np.isnan(X).any():
        cm = np.nanmean(X, axis=0)
        X[np.isnan(X)] = np.take(cm, np.where(np.isnan(X))[1])
    return X, y, names, feat_cols

X_train, y_train, train_names, feat_names = load_data(TRAIN_FEAT, TRAIN_LBLS, "Train")
X_test,  y_test,  test_names,  _          = load_data(TEST_FEAT,  TEST_LBLS,  "Test")

overlap = set(train_names) & set(test_names)
if overlap: raise ValueError(f"Leakage detected: {len(overlap)} shared galaxies!")
print("Leakage check passed.")


In [ ]:
scaler   = StandardScaler()
Xtr_s    = scaler.fit_transform(X_train)
Xte_s    = scaler.transform(X_test)

svm_base = SVC(kernel='rbf', C=78, gamma='scale',
               class_weight='balanced', probability=True, random_state=SEED)
svm_base.fit(Xtr_s, y_train)
print(f"SVM support vectors: class0={svm_base.n_support_[0]}, class1={svm_base.n_support_[1]}")


In [ ]:
# Cross-validation (manual folds so we can track per-fold)
skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)
cv_acc, cv_prec, cv_rec, cv_f1 = [], [], [], []
for fold, (ti, vi) in enumerate(skf.split(Xtr_s, y_train), 1):
    m = SVC(kernel='rbf', C=78, gamma='scale',
            class_weight='balanced', probability=True, random_state=SEED)
    m.fit(Xtr_s[ti], y_train[ti])
    yp = m.predict(Xtr_s[vi])
    cv_acc.append(accuracy_score(y_train[vi], yp))
    cv_prec.append(precision_score(y_train[vi], yp, zero_division=0))
    cv_rec.append(recall_score(y_train[vi], yp, zero_division=0))
    cv_f1.append(f1_score(y_train[vi], yp, zero_division=0))
    print(f"  Fold {fold}: acc={cv_acc[-1]:.4f} f1={cv_f1[-1]:.4f}")
print(f"\nCV accuracy: {np.mean(cv_acc):.4f} ± {np.std(cv_acc):.4f}")


In [ ]:
# Grid search
param_grid = {'C':[10,50,78,100,150], 'gamma':['scale','auto',0.001,0.01,0.1], 'kernel':['rbf','poly']}
gs = GridSearchCV(SVC(class_weight='balanced', probability=True, random_state=SEED),
                  param_grid,
                  cv=StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED),
                  scoring='accuracy', n_jobs=-1, verbose=0)
gs.fit(Xtr_s, y_train)
best_svm = gs.best_estimator_
print(f"Best params: {gs.best_params_}  CV acc: {gs.best_score_:.4f}")


In [ ]:
# Evaluation
def evaluate(model, X, y, names, label):
    yp = model.predict(X)
    yp_prob = model.predict_proba(X)[:,1]
    acc  = accuracy_score(y, yp)
    prec = precision_score(y, yp, zero_division=0)
    rec  = recall_score(y, yp, zero_division=0)
    f1   = f1_score(y, yp, zero_division=0)
    auc  = roc_auc_score(y, yp_prob)
    print(f"\n--- {label} ---")
    print(f"Accuracy={acc:.4f}  Precision={prec:.4f}  Recall={rec:.4f}  F1={f1:.4f}  AUC={auc:.4f}")
    print(classification_report(y, yp, target_names=['Unbarred','Barred'], digits=4, zero_division=0))
    return yp, yp_prob, accuracy_score(y,yp), confusion_matrix(y,yp), auc

_, __, ___, cm_train, _ = evaluate(best_svm, Xtr_s, y_train, train_names, "Training")
y_pred, y_prob, acc, cm, auc = evaluate(best_svm, Xte_s, y_test, test_names, "Test")


In [ ]:
# Permutation importance
perm = permutation_importance(best_svm, Xte_s, y_test, n_repeats=10, random_state=SEED, n_jobs=-1)
imp_df = pd.DataFrame({'Feature': feat_names,
                        'Importance': perm.importances_mean,
                        'Std': perm.importances_std}).sort_values('Importance', ascending=False)
print("Top 10 features:")
print(imp_df.head(10).to_string(index=False))


In [ ]:
# Plots
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

for ax, mat, title, cmap in [
    (axes[0,0], cm_train, 'Confusion – Train', 'Blues'),
    (axes[0,1], cm,       'Confusion – Test',  'Greens')]:
    sns.heatmap(mat, annot=True, fmt='d', cmap=cmap,
                xticklabels=['Unbarred','Barred'],
                yticklabels=['Unbarred','Barred'], ax=ax)
    ax.set_title(title); ax.set_ylabel('True'); ax.set_xlabel('Predicted')

fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[0,2].plot(fpr, tpr, lw=2, label=f'AUC={auc:.3f}')
axes[0,2].plot([0,1],[0,1],'k--'); axes[0,2].set_title('ROC')
axes[0,2].set_xlabel('FPR'); axes[0,2].set_ylabel('TPR'); axes[0,2].legend()

imp_df.head(10).plot.barh(x='Feature', y='Importance', ax=axes[1,0], legend=False)
axes[1,0].invert_yaxis(); axes[1,0].set_title('Feature Importance (Permutation)')

metrics = ['Accuracy','Precision','Recall','F1']
train_s = [accuracy_score(y_train, best_svm.predict(Xtr_s)),
           precision_score(y_train, best_svm.predict(Xtr_s), zero_division=0),
           recall_score(y_train, best_svm.predict(Xtr_s), zero_division=0),
           f1_score(y_train, best_svm.predict(Xtr_s), zero_division=0)]
test_s  = [acc, precision_score(y_test, y_pred, zero_division=0),
           recall_score(y_test, y_pred, zero_division=0), f1_score(y_test, y_pred, zero_division=0)]
x = np.arange(4)
axes[1,1].bar(x-0.2, train_s, 0.4, label='Train', alpha=0.8)
axes[1,1].bar(x+0.2, test_s,  0.4, label='Test',  alpha=0.8)
axes[1,1].set_xticks(x); axes[1,1].set_xticklabels(metrics, rotation=30)
axes[1,1].set_title('Metric Comparison'); axes[1,1].legend(); axes[1,1].set_ylim(0,1.05)

cv_m = [np.mean(cv_acc), np.mean(cv_prec), np.mean(cv_rec), np.mean(cv_f1)]
cv_s = [np.std(cv_acc),  np.std(cv_prec),  np.std(cv_rec),  np.std(cv_f1)]
axes[1,2].bar(metrics, cv_m, yerr=cv_s, capsize=5, alpha=0.8)
axes[1,2].set_title(f'{CV_FOLDS}-Fold CV'); axes[1,2].set_ylim(0,1.05)

plt.tight_layout()
plt.savefig('svm_results.png', dpi=300, bbox_inches='tight')
plt.show()
print("Saved svm_results.png")


In [ ]:
pd.DataFrame({
    'Target_Name': test_names, 'True_Label': y_test,
    'Predicted_Label': y_pred, 'Probability_Barred': y_prob,
    'Correct': y_test == y_pred,
}).to_csv('svm_predictions.csv', index=False)
imp_df.to_csv('svm_feature_importance.csv', index=False)
print("Saved: svm_predictions.csv, svm_feature_importance.csv")
